#### # 1. Initialize Libraries and Timeline Window Specification

In [0]:
from pyspark.sql import Window
import pyspark.sql.functions as F
from pyspark.sql.types import StringType



#### # 2. Extract Data and Standardize Baseline Date Type

In [0]:
# Load raw product info and cast start date from timestamp to plain date
df = (
    spark.table("workspace.bronze.crm_prd_info")
    .withColumn("prd_start_dt", F.col("prd_start_dt").cast("date"))
)

#### # 3. Calculate Contiguous Timeline End Dates via Lead Windowing

In [0]:
# Window specification to calculate continuous historical date ranges per product
product_window = Window.partitionBy("prd_key").orderBy("prd_start_dt")

# Calculate contiguous timeline end dates via Lead Windowing (with 9999-12-31 active record default)
# Replicates SQL logic: LEAD(prd_start_dt) OVER (...) - 1
df = df.withColumn(
    "prd_end_dt", 
    F.date_sub(F.lead(F.col("prd_start_dt"), 1).over(product_window), 1)
)

#### # 4. Extract Category Key and Strip Product Key Identifiers

In [0]:
# Extract category key and strip product key identifiers sequentially
df = (
    df.withColumn("cat_id", F.regexp_replace(F.substring(F.col("prd_key"), 1, 5), '-', '_'))
      .withColumn("prd_key", F.substring(F.col("prd_key"), 7, F.length(F.col("prd_key"))))
)

#### # 5. Handle Null Cost Elements and Standardize Product Lines

In [0]:
# Handle Null Cost elements and standardize product lines
df = (
    df.withColumn("prd_cost", F.coalesce(F.col("prd_cost"), F.lit(0)))
      .withColumn(
          "prd_line",
          F.when(F.upper(F.trim(F.col("prd_line"))) == "M", "Mountain")
           .when(F.upper(F.trim(F.col("prd_line"))) == "R", "Road")
           .when(F.upper(F.trim(F.col("prd_line"))) == "S", "Other Sales")
           .when(F.upper(F.trim(F.col("prd_line"))) == "T", "Touring")
           .otherwise("N/A")
      )
)

#### # 6. Apply Schema Renaming Map

In [0]:
# Apply schema renaming map
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#### # 7. Project Final Target Columns and Save to Silver Table

In [0]:
# Select target business keys, append system timestamps, and save the schema
silver_crm_prd_info_df = (
    df.select(
        "product_id",
        "category_id",
        "product_number",
        "product_name",
        "product_cost",
        "product_line",
        "start_date",
        "end_date"
    )
    .withColumn("dwh_create_date", F.current_timestamp())
)

(
    silver_crm_prd_info_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.crm_product")
)

silver_crm_prd_info_df.limit(10).display()